# Credit Risk Prediction — Model Training and Evaluation (v2)

**Author:** Aluka Precious Oluchukwu
**Project:** Credit Risk ML System — Model Training and Evaluation
**Version:** 2 — Corrected Pipeline Implementation

## Important Methodology Note

This notebook corrects a SMOTE data leakage issue identified in 
03_model_training.ipynb during peer review. The original notebook 
applied SMOTE to the full training set before cross validation — 
allowing synthetic samples to leak information into validation folds 
and inflating AUC-ROC estimates.

This corrected version uses an imblearn Pipeline that applies SMOTE 
exclusively inside each cross validation fold — ensuring validation 
folds contain only original unseen data and producing honest 
defensible performance estimates.

## Pipeline Architecture
```
Original Data → StratifiedKFold splits →
  Training fold: SMOTE applied internally → Model trains
  Validation fold: Original data only → Model evaluated
```

## Objectives

1. Honest cross validation comparison across all three models
2. Pipeline-based hyperparameter tuning without data leakage  
3. Final model selection based on defensible metrics
4. Evaluation on clean holdout test set
5. Threshold optimisation using business cost logic

In [1]:
# ─── Import libraries ──────────────────────────────────────────────────────────
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.model_selection import StratifiedKFold, cross_validate, RandomizedSearchCV
from sklearn.metrics import (accuracy_score, precision_score, recall_score,
                             f1_score, roc_auc_score, confusion_matrix,
                             classification_report, roc_curve)

from imblearn.pipeline import Pipeline
from imblearn.over_sampling import SMOTE

import warnings
warnings.filterwarnings("ignore")

print("Libraries imported sucessfully")

Libraries imported sucessfully


In [2]:
# ─── Load datasets ─────────────────────────────────────────────────────────────
# Original pre-SMOTE training data — used for Pipeline CV and tuning
X_train_orig = pd.read_csv("../data/processed/X_train_orig.csv")
y_train_orig = pd.read_csv("../data/processed/y_train_orig.csv").squeeze()

# Clean holdout test set — used only for final evaluation
X_test  = pd.read_csv("../data/processed/X_test.csv")
y_test  = pd.read_csv("../data/processed/y_test.csv").squeeze()

print("Datasets loaded successfully!")
print(f"\nTraining data — original imbalanced:")
print(f"  Shape: {X_train_orig.shape}")
print(f"  Good risk: {(y_train_orig == 0).sum()} ({(y_train_orig == 0).mean()*100:.1f}%)")
print(f"  Bad risk:  {(y_train_orig == 1).sum()} ({(y_train_orig == 1).mean()*100:.1f}%)")

print(f"\nTest data — clean holdout:")
print(f"  Shape: {X_test.shape}")
print(f"  Good risk: {(y_test == 0).sum()} ({(y_test == 0).mean()*100:.1f}%)")
print(f"  Bad risk:  {(y_test == 1).sum()} ({(y_test == 1).mean()*100:.1f}%)")

print("\nNote: SMOTE will be applied INSIDE each CV fold via Pipeline")
print("No synthetic data leakage into validation folds.")

Datasets loaded successfully!

Training data — original imbalanced:
  Shape: (800, 20)
  Good risk: 560 (70.0%)
  Bad risk:  240 (30.0%)

Test data — clean holdout:
  Shape: (200, 20)
  Good risk: 140 (70.0%)
  Bad risk:  60 (30.0%)

Note: SMOTE will be applied INSIDE each CV fold via Pipeline
No synthetic data leakage into validation folds.


## 2. Building imblearn Pipelines

Each Pipeline chains SMOTE and a classifier together as a single unit.
During cross validation the Pipeline guarantees SMOTE is applied only 
to the training fold — never the validation fold. This prevents data 
leakage and produces honest performance estimates.

The three Pipelines we evaluate:
1. Logistic Regression — interpretable regulatory baseline
2. Random Forest — primary ensemble candidate  
3. Gradient Boosting — advanced boosting candidate

In [3]:
# ─── Build imblearn Pipelines ──────────────────────────────────────────────────
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

pipelines = {
    'Logistic Regression': Pipeline([
        ('smote', SMOTE(random_state=42)),
        ('classifier', LogisticRegression(
            random_state=42, max_iter=1000))
    ]),
    'Random Forest': Pipeline([
        ('smote', SMOTE(random_state=42)),
        ('classifier', RandomForestClassifier(
            n_estimators=100, random_state=42))
    ]),
    'Gradient Boosting': Pipeline([
        ('smote', SMOTE(random_state=42)),
        ('classifier', GradientBoostingClassifier(
            n_estimators=100, random_state=42))
    ])
}

print("Pipelines built successfully!")
print("\nEach Pipeline structure:")
print("  Step 1 — SMOTE: applied to training fold only")
print("  Step 2 — Classifier: trains on SMOTE balanced fold")
print("\nPipelines ready for honest cross validation:")
for name in pipelines:
    print(f"  — {name}")

Pipelines built successfully!

Each Pipeline structure:
  Step 1 — SMOTE: applied to training fold only
  Step 2 — Classifier: trains on SMOTE balanced fold

Pipelines ready for honest cross validation:
  — Logistic Regression
  — Random Forest
  — Gradient Boosting


## 3. Honest Cross Validation — All Three Models

We run 5-fold Stratified Cross Validation using the imblearn Pipeline.
SMOTE is applied exclusively inside each training fold — the validation 
fold contains only original unseen data at all times.

This produces honest AUC-ROC estimates that are genuinely defensible
in production environments and regulatory reviews.

In [4]:
# ─── Honest Cross Validation ──────────────────────────────────────────────────
scoring = ['roc_auc', 'f1', 'recall', 'precision']

print("Running honest 5-fold Stratified Cross Validation...")
print("SMOTE applied inside each fold via Pipeline...")
print("=" * 75)
print(f"{'Model':<25} {'AUC-ROC':<20} {'F1':<12} {'Recall':<12} {'Precision':<12}")
print("-" * 75)

cv_results = {}

for name, pipeline in pipelines.items():
    scores = cross_validate(
        pipeline,
        X_train_orig,
        y_train_orig,
        cv=cv,
        scoring=scoring
    )

    cv_results[name] = {
        'AUC-ROC Mean': scores['test_roc_auc'].mean(),
        'AUC-ROC Std':  scores['test_roc_auc'].std(),
        'F1 Mean':      scores['test_f1'].mean(),
        'Recall Mean':  scores['test_recall'].mean(),
        'Precision Mean': scores['test_precision'].mean()
    }

    print(f"{name:<25} "
          f"{scores['test_roc_auc'].mean():.4f}"
          f"±{scores['test_roc_auc'].std():.3f}       "
          f"{scores['test_f1'].mean():.4f}      "
          f"{scores['test_recall'].mean():.4f}      "
          f"{scores['test_precision'].mean():.4f}")

print("=" * 75)

# ─── Gap analysis ─────────────────────────────────────────────────────────────
auc_scores = {k: v['AUC-ROC Mean'] for k, v in cv_results.items()}
best  = max(auc_scores, key=auc_scores.get)
worst = min(auc_scores, key=auc_scores.get)
gap   = auc_scores[best] - auc_scores[worst]

print(f"\nGap Analysis:")
print(f"  Best model:  {best} ({auc_scores[best]:.4f})")
print(f"  Worst model: {worst} ({auc_scores[worst]:.4f})")
print(f"  Gap:         {gap:.4f} ({gap*100:.1f}%)")

if gap > 0.04:
    print(f"\nVerdict: Gap > 4% — {best} selection is statistically justified")
else:
    print(f"\nVerdict: Gap < 4% — consider Logistic Regression for interpretability")

print("\nNote: These are HONEST estimates — SMOTE applied inside folds only")

Running honest 5-fold Stratified Cross Validation...
SMOTE applied inside each fold via Pipeline...
Model                     AUC-ROC              F1           Recall       Precision   
---------------------------------------------------------------------------
Logistic Regression       0.6756±0.035       0.4897      0.5458      0.4474
Random Forest             0.7632±0.037       0.5120      0.4833      0.5500
Gradient Boosting         0.7739±0.029       0.5659      0.5833      0.5534

Gap Analysis:
  Best model:  Gradient Boosting (0.7739)
  Worst model: Logistic Regression (0.6756)
  Gap:         0.0984 (9.8%)

Verdict: Gap > 4% — Gradient Boosting selection is statistically justified

Note: These are HONEST estimates — SMOTE applied inside folds only


## 4. Hyperparameter Tuning — Gradient Boosting Pipeline

Having confirmed Gradient Boosting as our champion model through honest 
cross validation we now tune its hyperparameters using RandomizedSearchCV 
with the same imblearn Pipeline architecture — ensuring SMOTE is applied 
correctly inside every tuning fold.

We optimise for AUC-ROC as our primary metric.

In [5]:
# ─── Hyperparameter Tuning via Pipeline ───────────────────────────────────────
from sklearn.model_selection import RandomizedSearchCV

# Parameter grid — prefixed with classifier__ to target the
# classifier step inside the Pipeline
param_grid = {
    'classifier__n_estimators':      [100, 200, 300, 500],
    'classifier__max_depth':         [3, 4, 5, 6, 7],
    'classifier__learning_rate':     [0.01, 0.05, 0.1, 0.15, 0.2],
    'classifier__min_samples_split': [2, 5, 10, 15],
    'classifier__min_samples_leaf':  [1, 2, 4, 6],
    'classifier__subsample':         [0.7, 0.8, 0.9, 1.0],
    'classifier__max_features':      ['sqrt', 'log2', None]
}

gb_pipeline = Pipeline([
    ('smote', SMOTE(random_state=42)),
    ('classifier', GradientBoostingClassifier(random_state=42))
])

random_search = RandomizedSearchCV(
    estimator=gb_pipeline,
    param_distributions=param_grid,
    n_iter=50,
    scoring='roc_auc',
    cv=cv,
    random_state=42,
    n_jobs=-1,
    verbose=1
)

print("Starting honest hyperparameter tuning...")
print("SMOTE applied inside each tuning fold via Pipeline...")
print("Testing 50 combinations × 5 folds = 250 fits")
print("Please wait — this will take a few minutes...\n")

random_search.fit(X_train_orig, y_train_orig)

print(f"\nBest parameters found:")
for param, value in random_search.best_params_.items():
    clean_param = param.replace('classifier__', '')
    print(f"  {clean_param}: {value}")

print(f"\nBest honest CV AUC-ROC: {random_search.best_score_:.4f}")
print(f"Default GB CV AUC-ROC:  0.7739")
print(f"Improvement:            {(random_search.best_score_ - 0.7739):.4f}")

Starting honest hyperparameter tuning...
SMOTE applied inside each tuning fold via Pipeline...
Testing 50 combinations × 5 folds = 250 fits
Please wait — this will take a few minutes...

Fitting 5 folds for each of 50 candidates, totalling 250 fits

Best parameters found:
  subsample: 0.8
  n_estimators: 300
  min_samples_split: 15
  min_samples_leaf: 2
  max_features: sqrt
  max_depth: 6
  learning_rate: 0.05

Best honest CV AUC-ROC: 0.7824
Default GB CV AUC-ROC:  0.7739
Improvement:            0.0085


In [9]:
# ─── Train final tuned Pipeline on full training data ─────────────────────────
best_pipeline = random_search.best_estimator_

# ─── Evaluate on clean holdout test set ───────────────────────────────────────
y_pred_final = best_pipeline.predict(X_test)
y_prob_final = best_pipeline.predict_proba(X_test)[:, 1]

print("=" * 65)
print("FINAL MODEL — TUNED GRADIENT BOOSTING PIPELINE")
print("Evaluated on clean holdout test set")
print("=" * 65)
print(f"Accuracy:  {accuracy_score(y_test, y_pred_final):.4f}")
print(f"Precision: {precision_score(y_test, y_pred_final):.4f}")
print(f"Recall:    {recall_score(y_test, y_pred_final):.4f}")
print(f"F1 Score:  {f1_score(y_test, y_pred_final):.4f}")
print(f"AUC-ROC:   {roc_auc_score(y_test, y_prob_final):.4f}")
print("=" * 65)

print("\nClassification Report:")
print(classification_report(y_test, y_pred_final,
      target_names=['Good Risk', 'Bad Risk']))

print("\nHonest CV AUC-ROC:  0.7824")
print(f"Test set AUC-ROC:   {roc_auc_score(y_test, y_prob_final):.4f}")
print(f"Generalisation gap: {abs(0.7824 - roc_auc_score(y_test, y_prob_final)):.4f}")
print("Small gap = model generalises well to unseen data")

FINAL MODEL — TUNED GRADIENT BOOSTING PIPELINE
Evaluated on clean holdout test set
Accuracy:  0.7700
Precision: 0.6296
Recall:    0.5667
F1 Score:  0.5965
AUC-ROC:   0.7869

Classification Report:
              precision    recall  f1-score   support

   Good Risk       0.82      0.86      0.84       140
    Bad Risk       0.63      0.57      0.60        60

    accuracy                           0.77       200
   macro avg       0.73      0.71      0.72       200
weighted avg       0.76      0.77      0.77       200


Honest CV AUC-ROC:  0.7824
Test set AUC-ROC:   0.7869
Generalisation gap: 0.0045
Small gap = model generalises well to unseen data


## Final Model Summary

| Attribute | Value |
|---|---|
| **Algorithm** | Gradient Boosting Classifier |
| **Pipeline** | SMOTE → Gradient Boosting |
| **Tuning Method** | RandomizedSearchCV — 50 iterations × 5 folds |
| **Honest CV AUC-ROC** | 0.7824 |
| **Test Set AUC-ROC** | 0.7869 |
| **Generalisation Gap** | 0.0045 — excellent stability |
| **Bad Risk Recall** | 56.7% |
| **Bad Risk Precision** | 62.9% |
| **Bad Risk F1** | 0.60 |
| **Data Leakage** | None — SMOTE applied inside Pipeline folds only |

## Methodology Note

The generalisation gap of 0.0045 between cross validation and test set 
AUC-ROC confirms the model is not overfitting and generalises reliably 
to unseen data. This is in contrast to our initial approach in 
03_model_training.ipynb where SMOTE applied before cross validation 
inflated CV AUC-ROC to 0.9184 — a misleading estimate caused by 
synthetic sample leakage across folds.

This corrected Pipeline implementation produces honest defensible 
metrics that accurately represent expected production performance.

In [10]:
# ─── Threshold optimisation ───────────────────────────────────────────────────
thresholds = [0.3, 0.35, 0.4, 0.45, 0.5]

print("=" * 70)
print("THRESHOLD ANALYSIS — TUNED GRADIENT BOOSTING PIPELINE")
print("=" * 70)
print(f"{'Threshold':<12} {'Accuracy':<12} {'Precision':<12} "
      f"{'Recall':<12} {'F1':<12} {'AUC-ROC':<12}")
print("-" * 70)

threshold_results = {}

for threshold in thresholds:
    y_pred_thresh = (y_prob_final >= threshold).astype(int)

    acc  = accuracy_score(y_test, y_pred_thresh)
    prec = precision_score(y_test, y_pred_thresh)
    rec  = recall_score(y_test, y_pred_thresh)
    f1   = f1_score(y_test, y_pred_thresh)
    auc  = roc_auc_score(y_test, y_prob_final)

    threshold_results[threshold] = {
        'Accuracy': acc, 'Precision': prec,
        'Recall': rec, 'F1': f1, 'AUC-ROC': auc
    }

    marker = ' ← default' if threshold == 0.5 else ''
    print(f"{threshold:<12} {acc:<12.4f} {prec:<12.4f} "
          f"{rec:<12.4f} {f1:<12.4f} {auc:<12.4f}{marker}")

print("=" * 70)
print("\nBusiness logic: Lower threshold → Higher Recall → Catch more bad risks")
print("Cost of missing bad risk >> Cost of declining good customer")

THRESHOLD ANALYSIS — TUNED GRADIENT BOOSTING PIPELINE
Threshold    Accuracy     Precision    Recall       F1           AUC-ROC     
----------------------------------------------------------------------
0.3          0.7350       0.5479       0.6667       0.6015       0.7869      
0.35         0.7400       0.5606       0.6167       0.5873       0.7869      
0.4          0.7550       0.5873       0.6167       0.6016       0.7869      
0.45         0.7600       0.6034       0.5833       0.5932       0.7869      
0.5          0.7700       0.6296       0.5667       0.5965       0.7869       ← default

Business logic: Lower threshold → Higher Recall → Catch more bad risks
Cost of missing bad risk >> Cost of declining good customer


In [11]:
# ─── Set optimal threshold ────────────────────────────────────────────────────
optimal_threshold = 0.30
y_pred_optimal = (y_prob_final >= optimal_threshold).astype(int)

print("=" * 65)
print("FINAL MODEL — OPTIMAL THRESHOLD 0.30")
print("=" * 65)
print(f"Accuracy:  {accuracy_score(y_test, y_pred_optimal):.4f}")
print(f"Precision: {precision_score(y_test, y_pred_optimal):.4f}")
print(f"Recall:    {recall_score(y_test, y_pred_optimal):.4f}")
print(f"F1 Score:  {f1_score(y_test, y_pred_optimal):.4f}")
print(f"AUC-ROC:   {roc_auc_score(y_test, y_prob_final):.4f}")
print("=" * 65)
print(classification_report(y_test, y_pred_optimal,
      target_names=['Good Risk', 'Bad Risk']))
print(f"Business interpretation:")
print(f"  This model correctly identifies "
      f"{recall_score(y_test, y_pred_optimal)*100:.1f}% "
      f"of all bad risk applicants")
print(f"  When it flags bad risk it is correct "
      f"{precision_score(y_test, y_pred_optimal)*100:.1f}% of the time")
print(f"\nPhase 4 — Model Training — COMPLETE!")
print(f"Champion model: Tuned Gradient Boosting Pipeline")
print(f"Optimal threshold: {optimal_threshold}")
print(f"Honest AUC-ROC: 0.7869")

FINAL MODEL — OPTIMAL THRESHOLD 0.30
Accuracy:  0.7350
Precision: 0.5479
Recall:    0.6667
F1 Score:  0.6015
AUC-ROC:   0.7869
              precision    recall  f1-score   support

   Good Risk       0.84      0.76      0.80       140
    Bad Risk       0.55      0.67      0.60        60

    accuracy                           0.73       200
   macro avg       0.70      0.72      0.70       200
weighted avg       0.75      0.73      0.74       200

Business interpretation:
  This model correctly identifies 66.7% of all bad risk applicants
  When it flags bad risk it is correct 54.8% of the time

Phase 4 — Model Training — COMPLETE!
Champion model: Tuned Gradient Boosting Pipeline
Optimal threshold: 0.3
Honest AUC-ROC: 0.7869
